In [61]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [62]:
import random
import numpy as np
import pandas as pd

In [63]:
random.seed(42)
np.random.seed(42)

METADATA_DIR = Path.home() / "Documents" / "interview-anonymization" / "data" / "metadata"

utterance_df = pd.read_parquet(
    METADATA_DIR / "voxceleb_utterances.parquet"
)

utterance_df.head()

,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
1,id10001,1zcIwhmdeo4,00003,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
2,id10001,1zcIwhmdeo4,00002,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
3,id10001,7gWzIy6yIIk,00004,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98
4,id10001,7gWzIy6yIIk,00001,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98


In [64]:
utterance_df.shape

(124843, 7)

In [65]:
utterance_df.groupby("split")["speaker_id"].nunique()

split
test     1166
train    1166
val      1166
Name: speaker_id, dtype: int64

In [66]:
counts = (
    utterance_df
    .groupby(["speaker_id", "split"])
    .size()
    .unstack(fill_value=0)
)

eligible_speakers = counts[
    (counts["train"] >= 20) &
    (counts["val"] >= 5) &
    (counts["test"] >= 5)
].index.tolist()

print("Eligible speakers:", len(eligible_speakers))

Eligible speakers: 818


In [67]:
random.seed(42)

selected_speakers = random.sample(
    eligible_speakers,
    100
)

subset_df = utterance_df[
    utterance_df["speaker_id"].isin(selected_speakers)
].copy()

In [68]:
limits = {
    "train": 20,
    "val": 5,
    "test": 5
}

sampled_parts = []

for split, limit in limits.items():
    part = (
        subset_df[subset_df["split"] == split]
        .groupby("speaker_id", group_keys=False)
        .sample(n=limit, random_state=42)
    )

    sampled_parts.append(part)

baseline_df = pd.concat(
    sampled_parts,
    ignore_index=True
)

In [69]:
baseline_df.groupby(
    ["split", "speaker_id"]
).size().groupby("split").describe()

,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,100.0,5.0,0.0,5.0,5.0,5.0,5.0,5.0
train,100.0,20.0,0.0,20.0,20.0,20.0,20.0,20.0
val,100.0,5.0,0.0,5.0,5.0,5.0,5.0,5.0


In [70]:
from src.ecapa import load_ecapa, extract_embedding

classifier = load_ecapa()

In [71]:
embeddings = []

n_total = len(baseline_df)

for i, path in enumerate(baseline_df["path"], start=1):
    z = extract_embedding(path, classifier)
    embeddings.append(z.numpy())

    if i % 100 == 0 or i == n_total:
        print(f"{i}/{n_total} embeddings extracted")

100/3000 embeddings extracted
200/3000 embeddings extracted
300/3000 embeddings extracted
400/3000 embeddings extracted
500/3000 embeddings extracted
600/3000 embeddings extracted
700/3000 embeddings extracted
800/3000 embeddings extracted
900/3000 embeddings extracted
1000/3000 embeddings extracted
1100/3000 embeddings extracted
1200/3000 embeddings extracted
1300/3000 embeddings extracted
1400/3000 embeddings extracted
1500/3000 embeddings extracted
1600/3000 embeddings extracted
1700/3000 embeddings extracted
1800/3000 embeddings extracted
1900/3000 embeddings extracted
2000/3000 embeddings extracted
2100/3000 embeddings extracted
2200/3000 embeddings extracted
2300/3000 embeddings extracted
2400/3000 embeddings extracted
2500/3000 embeddings extracted
2600/3000 embeddings extracted
2700/3000 embeddings extracted
2800/3000 embeddings extracted
2900/3000 embeddings extracted
3000/3000 embeddings extracted


In [72]:
Z = np.vstack(embeddings)

print("Z shape:", Z.shape)

assert Z.shape == (len(baseline_df), 192)
assert np.isfinite(Z).all()

print("All embeddings are valid.")

Z shape: (3000, 192)
All embeddings are valid.


In [73]:
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    EMBEDDING_DIR / "baseline_ecapa_embeddings.npy",
    Z
)

baseline_df.to_parquet(
    EMBEDDING_DIR / "baseline_metadata.parquet",
    index=False
)

print("Embeddings and metadata saved.")

Embeddings and metadata saved.


In [75]:
train_mask = baseline_df["split"].eq("train").to_numpy()
val_mask = baseline_df["split"].eq("val").to_numpy()
test_mask = baseline_df["split"].eq("test").to_numpy()

Z_train = Z[train_mask]
Z_val = Z[val_mask]
Z_test = Z[test_mask]

y_train = baseline_df.loc[
    train_mask, "speaker_id"
].to_numpy()

y_val = baseline_df.loc[
    val_mask, "speaker_id"
].to_numpy()

y_test = baseline_df.loc[
    test_mask, "speaker_id"
].to_numpy()

assert Z_train.shape == (2000, 192)
assert Z_val.shape == (500, 192)
assert Z_test.shape == (500, 192)

assert y_train.shape == (2000,)
assert y_val.shape == (500,)
assert y_test.shape == (500,)

In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import normalize

In [77]:
Z_train_norm = normalize(Z_train, norm="l2")
Z_val_norm = normalize(Z_val, norm="l2")
Z_test_norm = normalize(Z_test, norm="l2")

print(np.linalg.norm(Z_train_norm[0]))

0.99999994


In [78]:
model = LogisticRegression(
    C=1.0,
    max_iter=2000
)

model.fit(
    Z_train_norm,
    y_train
)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [79]:
y_val_pred = model.predict(Z_val_norm)

In [80]:
val_accuracy = accuracy_score(
    y_val,
    y_val_pred
)

print(
    f"Validation accuracy: {val_accuracy:.3f}"
)

Validation accuracy: 1.000


In [81]:
val_f1 = f1_score(
    y_val,
    y_val_pred,
    average="macro"
)

print(
    f"Validation macro-F1: {val_f1:.3f}"
)

Validation macro-F1: 1.000


In [82]:
C_values = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
]

validation_results = []

for C in C_values:
    model = LogisticRegression(
        C=C,
        max_iter=2000
    )

    model.fit(
        Z_train_norm,
        y_train
    )

    y_val_pred = model.predict(
        Z_val_norm
    )

    accuracy = accuracy_score(
        y_val,
        y_val_pred
    )

    macro_f1 = f1_score(
        y_val,
        y_val_pred,
        average="macro"
    )

    validation_results.append({
        "C": C,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    })


validation_results_df = pd.DataFrame(
    validation_results
)

validation_results_df

,C,accuracy,macro_f1
0,0.01,1.0,1.0
1,0.10,1.0,1.0
2,1.00,1.0,1.0
3,10.00,1.0,1.0
4,100.00,1.0,1.0


In [83]:
for C in C_values:
    model = LogisticRegression(
        C=C,
        max_iter=2000
    )

    model.fit(Z_train_norm, y_train)

    weight_norm = np.linalg.norm(model.coef_)

    print(f"C={C:<6} ||W||={weight_norm:.3f}")

C=0.01   ||W||=1.616
C=0.1    ||W||=15.539
C=1.0    ||W||=62.509
C=10.0   ||W||=94.960
C=100.0  ||W||=127.637


In [84]:
for speaker in selected_speakers:
    speaker_df = baseline_df[
        baseline_df["speaker_id"] == speaker
    ]

    train_videos = set(
        speaker_df.loc[
            speaker_df["split"] == "train",
            "video_id"
        ]
    )

    val_videos = set(
        speaker_df.loc[
            speaker_df["split"] == "val",
            "video_id"
        ]
    )

    test_videos = set(
        speaker_df.loc[
            speaker_df["split"] == "test",
            "video_id"
        ]
    )

    assert train_videos.isdisjoint(val_videos)
    assert train_videos.isdisjoint(test_videos)
    assert val_videos.isdisjoint(test_videos)

assert baseline_df["path"].is_unique

print("No video or utterance leakage.")

No video or utterance leakage.


In [85]:
best_C = 0.01
Z_trainval_norm = np.vstack([
    Z_train_norm,
    Z_val_norm
])

y_trainval = np.concatenate([
    y_train,
    y_val
])

final_model = LogisticRegression(
    C=best_C,
    max_iter=2000
)

final_model.fit(
    Z_trainval_norm,
    y_trainval
)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use 

In [89]:
y_test_pred = final_model.predict(
    Z_test_norm
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

print(f"Test accuracy: {test_accuracy:.3f}")
print(f"Test macro-F1: {test_f1:.3f}")
print(f"Chance level: {1 / 100:.3f}")

Test accuracy: 1.000
Test macro-F1: 1.000
Chance level: 0.010


In [87]:
rng = np.random.default_rng(42)

y_train_shuffled = rng.permutation(y_train)

shuffled_model = LogisticRegression(
    C=1.0,
    max_iter=2000
)

shuffled_model.fit(
    Z_train_norm,
    y_train_shuffled
)

y_val_shuffled_pred = shuffled_model.predict(
    Z_val_norm
)

shuffled_accuracy = accuracy_score(
    y_val,
    y_val_shuffled_pred
)

print("Accuracy with shuffled labels:", shuffled_accuracy)

Accuracy with shuffled labels: 0.01


In [88]:
similarities = Z_val_norm @ Z_train_norm.T

nearest_idx = similarities.argmax(axis=1)

nn_predictions = y_train[nearest_idx]

nn_accuracy = accuracy_score(
    y_val,
    nn_predictions
)

print("1-NN cosine accuracy:", nn_accuracy)

1-NN cosine accuracy: 1.0
